# MNIST Digit Embeddings

MNIST is a useful sandbox for Glass Box UMAP because each input feature is a pixel with a known spatial location. This means we can visualize feature contributions as heatmaps overlaid on the 28x28 image grid, making it immediately obvious whether the model is attending to sensible parts of each digit.

In this tutorial, we train a Glass Box UMAP on MNIST handwritten digits, then use Jacobian-based contributions to produce per-digit importance maps that show which pixels most strongly influence each digit's position in the embedding.

## Load Data

We load a subset of MNIST using scikit-learn. Each sample is a 784-dimensional vector (28x28 pixels flattened), with pixel values in [0, 255].

In [1]:
import numpy as np
import torch
from sklearn.datasets import fetch_openml

data, target = fetch_openml("mnist_784", version=1, return_X_y=True, as_frame=False)

N_SAMPLES = 4000
X = torch.from_numpy(data[:N_SAMPLES].astype(np.float32))
y = target[:N_SAMPLES].astype(int)

print(f"X: shape {X.shape}, dtype {X.dtype}")
print(f"y: shape {y.shape}, unique digits {np.unique(y)}")

X: shape torch.Size([4000, 784]), dtype torch.float32
y: shape (4000,), unique digits [0 1 2 3 4 5 6 7 8 9]


## Fit Glass Box UMAP

We fit a Glass Box UMAP with PCA preprocessing. Even though MNIST is only 784-dimensional (much smaller than a typical gene expression matrix), PCA still helps by denoising and speeding up training. As with the gene expression tutorial, contributions are projected back through PCA so the final results are in terms of individual pixels.

In [2]:
from glass_box_umap import GlassBoxUMAP

reducer = GlassBoxUMAP(
    epochs=100,
    lr=1e-3,
    batch_size=256,
    repulsion_strength=1.0,
    pca_components=25,
    random_state=42,
)

reducer.fit(X)
embedding = reducer.transform(X)

/Users/evan/Software/arcadia/glass-box-umap/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Seed set to 42
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores


Thu Apr 16 16:28:45 2026 Building RP forest with 8 trees
Thu Apr 16 16:28:46 2026 NN descent for 12 iterations
	 1  /  12
	 2  /  12
	 3  /  12
	 4  /  12
	 5  /  12
	Stopping threshold met -- exiting after 5 iterations


/Users/evan/Software/arcadia/glass-box-umap/.venv/lib/python3.13/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /private/var/folders/v4/q0wyvpsj4h901lfkjcln68nw0000gn/T/tmpumok8z1b exists and is not empty.


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type         ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ encoder │ DeepPReLUNet │  270 K │ train │     0 │
└───┴─────────┴──────────────┴────────┴───────┴───────┘

Trainable params: 270 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 270 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 23                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/Users/evan/Software/arcadia/glass-box-umap/.venv/lib/python3.13/site-packages/rich/live.py:256: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/Users/evan/Software/arcadia/glass-box-umap/.venv/lib/python3.13/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.
`Trainer.fit` stopped: `max_epochs=100` reached.


## Plot Embedding

We plot the embedding colored by digit label to verify the model has learned a reasonable structure.

In [ ]:
from glass_box_umap.plotting import plot_embedding

digit_names = [str(d) for d in range(10)]
fig = plot_embedding(Z=embedding, group_ids=y, group_names=digit_names)
fig.show()

## Compute feature contributions

`compute_contributions` returns a `(n_samples, n_components, n_features)` array: for each sample, for each embedding dimension, the contribution of every pixel. To get a single importance score per pixel per sample, we take the L2 norm across the two embedding components using `reduce_contributions`.

We then average these importance scores within each digit class and reshape back to 28x28 to produce a spatial heatmap. Bright regions are the pixels that most strongly influence where that digit ends up in the embedding. For example, you might expect the central stroke of a "1" to light up, or the loops of an "8".

In [ ]:
import matplotlib.pyplot as plt
from glass_box_umap.jacobian import reduce_contributions

contributions = reducer.compute_contributions(X)
importance = reduce_contributions(contributions)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for digit in range(10):
    ax = axes[digit // 5, digit % 5]
    mask = y == digit
    mean_importance = importance[mask].mean(axis=0).reshape(28, 28)
    ax.imshow(mean_importance, cmap="magma")
    ax.set_title(f"Digit {digit}")
    ax.axis("off")

fig.suptitle("Mean pixel importance per digit", fontsize=14)
fig.tight_layout()
fig.show()

## What's next

The heatmaps above show *average* importance per digit class, but the full contribution array is per-sample. This means you can inspect individual digits to understand outliers -- for example, a "9" that the model places near the "4" cluster might show high importance in the vertical stroke (shared by both digits) and low importance in the upper loop (which distinguishes them).